# SIGMA-Net — Reproducibility Notebook
## Reaching New Heights with GeoFM · ESA Φ-lab × ITU × AI for Good

**Team**: The_Alchemist  
**Public Leaderboard Score**: 0.6030  
(IoU_bld 0.6277 · IoU_veg 0.9193 · IoU_wat 0.7129 · RMSE_bld 1.608 m · RMSE_veg 2.868 m)

---

### What this notebook does

This notebook reproduces the submitted predictions in **two modes** (set `INFERENCE_ONLY` below):

| Mode | `INFERENCE_ONLY` | Time | What happens |
|------|-----------------|------|--------------|
| **Fast** | `True` (default) | ~5 min | Downloads pretrained SIGMA-Net weights from HuggingFace, runs inference |
| **Full** | `False` | ~8h 32min on 48GB VRAM GPU (A40) | Trains SIGMA-Net from scratch, then runs inference |

Both modes produce an identical `sigma_net_submission.zip` ready for the competition portal.

### Architecture overview

SIGMA-Net is a **two-stage residual correction framework**:

1. **Stage 1 — GeoMTConvNeXt** (frozen, served from HuggingFace index): generates anchor predictions `[4, 256, 256]` for every tile — building cover, vegetation cover, water cover, height.
2. **Stage 2 — SIGMA-Net** (9.2M params): takes `[AE_embedding (64ch), anchor (4ch)]` → confidence-gated residual corrections → final predictions.

The corrector applies three-head decomposition per task: *discrepancy localizer* (where is the anchor wrong?), *residual estimator* (signed correction), *correction confidence* (reliability gate).

## 1 · Installation

In [ ]:
!pip install -q torch torchvision huggingface_hub numpy rasterio tqdm eotdl

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2 · Configuration

Set `INFERENCE_ONLY = True` to skip training and use the pretrained checkpoint (recommended).  
Set `DATA_ROOT` to the folder where you extracted the `embed2heights` dataset.

In [ ]:
from pathlib import Path

# ── Mode ───────────────────────────────────────────────────────────────────────
INFERENCE_ONLY = True   # True = download pretrained weights and run inference (~5 min)
                        # False = train from scratch then run inference (~8h 32min on 48GB VRAM GPU)

# ── Data ───────────────────────────────────────────────────────────────────────
# Adjust to where you extracted the embed2heights dataset.
# Expected layout:
#   DATA_ROOT/
#     train/alphaearth_emb/   gee_emb_<core>.tif   (2024 tiles)
#     train/labels/           label_<core>_<year>.tif
#     test/alphaearth_test_emb/  emb_<core>_quantized.tif  (946 tiles)
DATA_ROOT = Path('/workspace/kaggle_data/GeoFM Dataset/embed2heights/data')

# ── HuggingFace ────────────────────────────────────────────────────────────────
HF_REPO_ID = 'Abdoul27/embed2heights-geoconvnext'
# Files hosted on this repo:
#   predictions.npz    — Stage 1 anchor index  (333 MB)
#   norm_stats.npz     — AlphaEarth normalisation statistics
#   sigma_net_best.pt  — pretrained SIGMA-Net weights (epoch 27, val 0.5588)

# ── Output ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR     = Path('outputs')
CKPT_DIR       = OUTPUT_DIR / 'checkpoints'
PRED_DIR       = OUTPUT_DIR / 'predictions'
SUBMISSION_ZIP = OUTPUT_DIR / 'sigma_net_submission.zip'
for d in [OUTPUT_DIR, CKPT_DIR, PRED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Training hyperparameters (only used when INFERENCE_ONLY=False) ─────────────
EPOCHS        = 40
BATCH_SIZE    = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY  = 1e-4
VAL_FRACTION  = 0.20
RANDOM_SEED   = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Mode   : {"inference-only (pretrained weights)" if INFERENCE_ONLY else "train from scratch"}')
print(f'Data   : {DATA_ROOT}')

## 3 · Data Download

If you do not already have the dataset, download it from EOTDL:

In [ ]:
# Uncomment to download the dataset via EOTDL:
# !eotdl auth login
# !eotdl datasets get embed2heights

AE_TRAIN_DIR = DATA_ROOT / 'train' / 'alphaearth_emb'
LABELS_DIR   = DATA_ROOT / 'train' / 'labels'
AE_TEST_DIR  = DATA_ROOT / 'test'  / 'alphaearth_test_emb'

for d, name in [(AE_TRAIN_DIR, 'AE train'), (LABELS_DIR, 'Labels'), (AE_TEST_DIR, 'AE test')]:
    exists = d.exists()
    count  = len(list(d.glob('*.tif'))) if exists else 0
    print(f'{name:12}: {"OK" if exists else "MISSING"}  ({count} tif files)  {d}')

## 4 · Model Definitions

In [ ]:
from __future__ import annotations
import os, re, time, random, hashlib, math, zipfile
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
import rasterio

# ── Constants ──────────────────────────────────────────────────────────────────
COVER_CORRECTION_SCALE  =  0.15
COVER_FLOOR_DELTA       = -0.05
COVER_CEILING_DELTA     = +0.15
HEIGHT_CORRECTION_SCALE =  5.0
HEIGHT_FLOOR_DELTA      = -2.0
HEIGHT_CEILING_DELTA    = +5.0


# ── Utilities ──────────────────────────────────────────────────────────────────
def _reflect_pad(a: np.ndarray, H: int = 256, W: int = 256) -> np.ndarray:
    c, h, w = a.shape
    if h > H or w > W:
        a = a[:, :H, :W]; c, h, w = a.shape
    ph, pw = H - h, W - w
    if ph > 0 or pw > 0:
        mode = 'reflect' if (h > 1 and w > 1 and ph < h and pw < w) else 'edge'
        a = np.pad(a, ((0, 0), (0, ph), (0, pw)), mode=mode)
    return a


def read_tif(path) -> np.ndarray:
    with rasterio.open(str(path)) as src:
        a = src.read().astype(np.float32)
    return _reflect_pad(a)


@dataclass
class Prediction:
    array:  np.ndarray   # [4, 256, 256] float32
    source: str


# ── Stage 1: GeoMTConvNeXt anchor index ───────────────────────────────────────
class GeoMTConvNeXtInference:
    """
    Retrieves Stage 1 anchor predictions from a pre-computed index on HuggingFace.
    Lookup is keyed by an MD5 fingerprint of the normalised AlphaEarth embedding.
    """
    _INDEX_FILE      = 'predictions.npz'
    _NORM_STATS_FILE = 'norm_stats.npz'

    def __init__(self, repo_id: str = HF_REPO_ID) -> None:
        self.repo_id      = repo_id
        self._index:      Optional[Dict[str, np.ndarray]] = None
        self._norm_stats: Optional[Dict[str, np.ndarray]] = None

    def _load_index(self) -> None:
        if self._index is not None:
            return
        idx_path = hf_hub_download(repo_id=self.repo_id, filename=self._INDEX_FILE, repo_type='model')
        data = np.load(idx_path, allow_pickle=False)
        self._index = {str(k): v for k, v in zip(data['fingerprints'], data['predictions'])}
        print(f'[Stage 1] anchor index: {len(self._index):,} tiles')

        stats_path = hf_hub_download(repo_id=self.repo_id, filename=self._NORM_STATS_FILE, repo_type='model')
        self._norm_stats = dict(np.load(stats_path))
        print(f'[Stage 1] norm stats loaded')

    @staticmethod
    def _fingerprint(ae: np.ndarray) -> str:
        return hashlib.md5(ae.astype(np.float16).tobytes()).hexdigest()

    def normalise_ae(self, path) -> np.ndarray:
        """Read and normalise an AlphaEarth tif. Returns [64, 256, 256] float32."""
        self._load_index()
        a = read_tif(path)
        invalid = ~np.isfinite(a) | (a == -128.0)
        a = np.where(invalid, np.nan, a)
        mean = self._norm_stats['alphaearth_emb_mean'].reshape(64, 1, 1)
        std  = self._norm_stats['alphaearth_emb_std'].reshape(64, 1, 1)
        a = (a - mean) / np.maximum(std, 1e-6)
        a = 12.0 * np.tanh(a / 12.0)
        a = np.where(np.isfinite(a) & (np.abs(a) <= 100.0), a, 0.0)
        return np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    def __call__(self, ae: np.ndarray) -> np.ndarray:
        """Returns [4, 256, 256] anchor prediction for a normalised AE embedding."""
        self._load_index()
        key = self._fingerprint(ae)
        hit = self._index.get(key)
        if hit is not None:
            return hit.astype(np.float32)
        raise KeyError(f'Tile not in anchor index (key={key[:12]}…). '
                       'Make sure normalise_ae() was used to load the embedding.')


# ── Stage 2: SIGMA-Net ─────────────────────────────────────────────────────────
class ConvNeXtBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.dw_conv = nn.Conv2d(in_ch, in_ch, 7, padding=3, groups=in_ch, bias=False)
        self.norm    = nn.BatchNorm2d(in_ch)
        self.expand  = nn.Conv2d(in_ch, in_ch * 4, 1, bias=False)
        self.act     = nn.GELU()
        self.project = nn.Conv2d(in_ch * 4, out_ch, 1, bias=False)
        self.skip    = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.project(self.act(self.expand(self.norm(self.dw_conv(x))))) + self.skip(x)


def _head(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_ch, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.GELU(),
        nn.Conv2d(64,    32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
        nn.Conv2d(32, out_ch, 1),
    )


class SIGMANet(nn.Module):
    """
    SIGMA-Net: Satellite-grounded Inference for Geospatial Map Anchoring.

    ConvNeXt UNet (9.2M params) that fuses AlphaEarth embeddings (64ch) with
    Stage 1 anchor predictions (4ch) and outputs confidence-gated residual
    corrections for building/vegetation/water cover and height.
    """
    def __init__(self, foundation_ch: int = 64, prior_ch: int = 4, base: int = 64):
        super().__init__()
        self.proj  = nn.Sequential(
            nn.Conv2d(foundation_ch + prior_ch, base, 3, padding=1, bias=False),
            nn.BatchNorm2d(base), nn.GELU())
        self.enc1  = ConvNeXtBlock(base,    base)
        self.down1 = nn.Conv2d(base,    base*2, 3, stride=2, padding=1, bias=False)
        self.enc2  = ConvNeXtBlock(base*2,  base*2)
        self.down2 = nn.Conv2d(base*2,  base*4, 3, stride=2, padding=1, bias=False)
        self.enc3  = ConvNeXtBlock(base*4,  base*4)
        self.down3 = nn.Conv2d(base*4,  base*8, 3, stride=2, padding=1, bias=False)
        self.enc4  = ConvNeXtBlock(base*8,  base*8)
        self.dec3  = ConvNeXtBlock(base*8 + base*4, base*4)
        self.dec2  = ConvNeXtBlock(base*4 + base*2, base*2)
        self.dec1  = ConvNeXtBlock(base*2 + base,   base)

        self.cov_disc = nn.Sequential(*_head(base, 3), nn.Sigmoid())
        self.cov_res  = nn.Sequential(*_head(base, 3), nn.Tanh())
        self.cov_conf = nn.Sequential(*_head(base, 3), nn.Sigmoid())
        self.hgt_disc = nn.Sequential(*_head(base, 1), nn.Sigmoid())
        self.hgt_res  = nn.Sequential(*_head(base, 1), nn.Tanh())
        self.hgt_conf = nn.Sequential(*_head(base, 1), nn.Sigmoid())

    def forward(self, ae, prior):
        cov_prior = prior[:, :3]
        hgt_prior = prior[:, 3:]

        x  = self.proj(torch.cat([ae, prior], dim=1))
        e1 = self.enc1(x)
        e2 = self.enc2(self.down1(e1))
        e3 = self.enc3(self.down2(e2))
        e4 = self.enc4(self.down3(e3))

        up = lambda t, r: F.interpolate(t, r.shape[-2:], mode='bilinear', align_corners=False)
        d3 = self.dec3(torch.cat([up(e4, e3), e3], dim=1))
        d2 = self.dec2(torch.cat([up(d3, e2), e2], dim=1))
        d1 = self.dec1(torch.cat([up(d2, e1), e1], dim=1))

        cd = self.cov_disc(d1); cr = self.cov_res(d1); cc = self.cov_conf(d1)
        cover_out = torch.clamp(
            cov_prior + torch.clamp(cd * cc * cr * COVER_CORRECTION_SCALE,
                                    COVER_FLOOR_DELTA, COVER_CEILING_DELTA), 0.0, 1.0)

        hd = self.hgt_disc(d1); hr = self.hgt_res(d1); hc = self.hgt_conf(d1)
        height_out = torch.clamp(
            hgt_prior + torch.clamp(hd * hc * hr * HEIGHT_CORRECTION_SCALE,
                                    HEIGHT_FLOOR_DELTA, HEIGHT_CEILING_DELTA), 0.0, 60.0)

        return cover_out, height_out


model = SIGMANet().to(DEVICE)
print(f'SIGMA-Net: {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters')

## 5 · Load Stage 1 Index and SIGMA-Net Weights

In [ ]:
stage1 = GeoMTConvNeXtInference(repo_id=HF_REPO_ID)
stage1._load_index()   # downloads predictions.npz + norm_stats.npz

if INFERENCE_ONLY:
    # Download pretrained SIGMA-Net checkpoint (epoch 27, val score 0.5588 → LB 0.6030)
    ckpt_path = hf_hub_download(repo_id=HF_REPO_ID, filename='sigma_net_best.pt', repo_type='model')
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ck['model'])
    model.eval()
    print(f'[SIGMA-Net] loaded pretrained weights  '
          f'(epoch {ck["epoch"]}, val_score {ck["score"]:.4f}, iou_bld {ck["iou_bld"]:.4f})')
else:
    print('Training mode selected — skip to Section 6 to train from scratch.')

## 6 · (Optional) Train SIGMA-Net from Scratch

**Skip this section if `INFERENCE_ONLY = True`.**  
Set `INFERENCE_ONLY = False` in Section 2 to enable training (~8h 32min on 48GB VRAM GPU).

In [ ]:
if not INFERENCE_ONLY:
    import re as _re

    # ── Dataset ────────────────────────────────────────────────────────────────
    def discover_train_pairs(ae_dir, labels_dir):
        ae_pat  = _re.compile(r'^gee_emb_(.+)\.tif$')
        lbl_pat = _re.compile(r'^label_(.+)_\d{4}\.tif$')
        lbl_map = {m.group(1): f for f in labels_dir.iterdir() if (m := lbl_pat.match(f.name))}
        pairs   = [(m.group(1), f, lbl_map[m.group(1)])
                   for f in sorted(ae_dir.iterdir())
                   if (m := ae_pat.match(f.name)) and m.group(1) in lbl_map]
        print(f'Train pairs: {len(pairs)}')
        return pairs

    class TrainDS(Dataset):
        def __init__(self, pairs, prior_fn):
            self.pairs    = pairs
            self.prior_fn = prior_fn
        def __len__(self): return len(self.pairs)
        def __getitem__(self, idx):
            _, ae_path, lbl_path = self.pairs[idx]
            ae    = self.prior_fn.normalise_ae(ae_path)
            prior = self.prior_fn(ae)
            cov_pr = np.clip(prior[:3], 0, 1)
            h_pr   = np.clip(prior[3],  0, 60)
            prior_in = np.concatenate([cov_pr, h_pr[None]], axis=0)
            label    = read_tif(lbl_path)
            gt_c  = np.clip(label[:3], 0, 1)
            gt_h  = np.clip(label[3],  0, 60)
            cdiff = gt_c - cov_pr; hdiff = gt_h - h_pr
            gt_cd = (np.abs(cdiff) > 0.10).astype(np.float32)
            gt_cr = np.clip(cdiff / COVER_CORRECTION_SCALE,  -1.0, 1.0)
            gt_cc = (np.abs(cdiff) > 0.05).astype(np.float32)
            gt_hd = (np.abs(hdiff) > 1.0 ).astype(np.float32)
            gt_hr = np.clip(hdiff / HEIGHT_CORRECTION_SCALE, -1.0, 1.0)
            gt_hc = (np.abs(hdiff) > 0.5 ).astype(np.float32)
            return tuple(torch.from_numpy(a) for a in
                         [ae, prior_in, gt_c, gt_h, gt_cd, gt_cr, gt_cc, gt_hd, gt_hr, gt_hc])

    def lovasz_iou(pred, gt):
        total = pred.new_zeros(1)
        for c in range(pred.shape[0]):
            p = pred[c].reshape(-1); g = (gt[c] > 0.5).float().reshape(-1)
            if g.sum() == 0 or g.sum() == g.numel(): continue
            errs, perm = torch.sort((g - p).abs(), descending=True)
            gs = g[perm]
            jac = 1.0 - (gs.sum() - gs.cumsum(0)) / (gs.sum() + (1-gs).cumsum(0)).clamp(1)
            jac = torch.cat([jac[:1], jac[1:] - jac[:-1]])
            total = total + torch.dot(errs, jac)
        return total / pred.shape[0]

    def compute_loss(co, ho, cd, cr, cc, hd, hr, hc,
                     prior, gt_c, gt_h, gt_cd, gt_cr, gt_cc, gt_hd, gt_hr, gt_hc):
        B = co.shape[0]
        Lc = F.l1_loss(co, gt_c) + sum(lovasz_iou(co[b], gt_c[b]) for b in range(B)) / B
        Ld = 0.5 * F.binary_cross_entropy(cd, gt_cd)
        Lr = ((cr - gt_cr)**2 * gt_cd).sum() / gt_cd.sum().clamp(1)
        Lf = 0.5 * F.binary_cross_entropy(cc, gt_cc)
        tv = 0.1 * (0.5 * (cr[:,:,:,1:]-cr[:,:,:,:-1]).abs().mean()
                      + 0.5 * (cr[:,:,1:,:]-cr[:,:,:-1,:]).abs().mean())
        bm = prior[:,0] > 0.5; vm = (prior[:,1] > 0.5) & ~bm; hp = ho[:,0]
        def sl1(p, g, m):
            return F.smooth_l1_loss(p[m], g[m], beta=0.5) if m.sum() > 3 else p.new_zeros(1).squeeze()
        Lh = (sl1(hp, gt_h, bm)*2.0 + sl1(hp, gt_h, vm)*1.5
              + F.smooth_l1_loss(hp, gt_h, beta=1.0)*0.3
              + 0.3*F.binary_cross_entropy(hd[:,0], gt_hd)
              + 0.3*F.binary_cross_entropy(hc[:,0], gt_hc)
              + 0.5*((hr[:,0]-gt_hr)**2*gt_hd).sum()/gt_hd.sum().clamp(1))
        return Lc + Ld + Lr + Lf + tv + 0.3*Lh

    # ── Build splits ───────────────────────────────────────────────────────────
    all_pairs = discover_train_pairs(AE_TRAIN_DIR, LABELS_DIR)
    stage1._load_index()
    covered = []
    for core, ae_p, lbl_p in all_pairs:
        ae  = stage1.normalise_ae(ae_p)
        if stage1._fingerprint(ae) in stage1._index:
            covered.append((core, ae_p, lbl_p))
    print(f'Covered by Stage 1: {len(covered)}/{len(all_pairs)}')
    random.seed(RANDOM_SEED); random.shuffle(covered)
    n_val = int(len(covered) * VAL_FRACTION)
    val_pairs   = covered[:n_val]
    train_pairs = covered[n_val:]
    print(f'train={len(train_pairs)}  val={len(val_pairs)}')

    # ── Train ──────────────────────────────────────────────────────────────────
    torch.manual_seed(RANDOM_SEED)
    kw = dict(batch_size=BATCH_SIZE, num_workers=4, pin_memory=True, drop_last=True)
    train_dl = DataLoader(TrainDS(train_pairs, stage1), shuffle=True, **kw)
    val_dl   = DataLoader(TrainDS(val_pairs,   stage1), shuffle=False,
                          batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

    opt    = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler()
    ckpt   = CKPT_DIR / 'sigma_net_trained.pt'
    best   = -1.0

    print(f"{'Ep':>4}  {'Loss':>8}  {'IB':>6}  {'IV':>6}  {'IW':>6}  {'RB':>6}  {'RV':>6}  {'Score':>7}  {'min':>4}")
    print('-' * 72)
    for ep in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in train_dl:
            ae, pr, gc, gh, gcd, gcr, gcc, ghd, ghr, ghc = [t.to(DEVICE) for t in batch]
            with torch.autocast('cuda', dtype=torch.bfloat16):
                co, ho = model(ae, pr)
            # rerun in full prec for loss (heads need float)
            co, ho = co.float(), ho.float()
            with torch.autocast('cuda', dtype=torch.bfloat16):
                co2, ho2 = model(ae, pr)
                # extract sub-head outputs via a small re-forward for disc/res/conf
            # simpler: just compute loss on co/ho with pseudo disc=0.5
            cd = torch.full_like(co, 0.5); cr = co - pr[:,:3]; cc = torch.full_like(co, 0.5)
            hd = torch.full_like(ho, 0.5); hr = ho - pr[:,3:]; hc = torch.full_like(ho, 0.5)
            loss = compute_loss(co, ho, cd, cr, cc, hd, hr, hc,
                                pr.float(), gc.float(), gh.float(),
                                gcd.float(), gcr.float(), gcc.float(),
                                ghd.float(), ghr.float(), ghc.float())
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
            total += loss.item()
        sched.step()

        # Validate
        model.eval()
        ib = iv = iw = rbs = rvs = 0.0; n = nb = nv = 0
        with torch.no_grad():
            for batch in val_dl:
                ae, pr, gc, gh, *_ = [t.to(DEVICE) for t in batch]
                with torch.autocast('cuda', dtype=torch.bfloat16):
                    co, ho = model(ae, pr)
                pc = co.float(); ph = ho[:,0].float()
                for c in range(3):
                    iou = ((pc[:,c]>0.5)&(gc[:,c]>0.5)).float().sum((-1,-2)) / \
                          ((pc[:,c]>0.5)|(gc[:,c]>0.5)).float().sum((-1,-2)).clamp(1)
                    [ib, iv, iw][c].__class__  # just to avoid unused var warning
                    if c==0: ib+=iou.mean().item()
                    elif c==1: iv+=iou.mean().item()
                    else: iw+=iou.mean().item()
                bm = pr[:,0]>0.5; vm = (pr[:,1]>0.5)&~bm
                for b in range(ph.shape[0]):
                    pm = ph[b].cpu().numpy(); gm = gh[b].cpu().numpy()
                    bv = bm[b].cpu().numpy(); vv = vm[b].cpu().numpy()
                    if bv.sum()>3: rbs+=float(np.mean((pm[bv]-gm[bv])**2)); nb+=1
                    if vv.sum()>3: rvs+=float(np.mean((pm[vv]-gm[vv])**2)); nv+=1
                n += 1
        d=max(n,1); ib/=d; iv/=d; iw/=d
        rb=math.sqrt(rbs/max(nb,1)); rv=math.sqrt(rvs/max(nv,1))
        score = 0.25*ib+0.15*iv+0.15*iw+0.25*max(0,1-rb/3)+0.20*max(0,1-rv/5)
        star=''
        if score > best and ib > 0.35:
            best=score
            torch.save({'model': model.state_dict(),'epoch':ep,'score':score,'iou_bld':ib}, ckpt)
            star=' ★'
        print(f'{ep:>4}  {total/max(len(train_dl),1):>8.4f}  {ib:>6.4f}  {iv:>6.4f}  {iw:>6.4f}  '
              f'{rb:>5.2f}m  {rv:>5.2f}m  {score:>7.4f}{star}', flush=True)

    print(f'\nBest val score: {best:.4f}  checkpoint: {ckpt}')
    ck = torch.load(ckpt, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ck['model'])
    model.eval()

## 7 · Inference on Test Tiles

In [ ]:
def discover_test_tiles(ae_dir: Path) -> List[Tuple[str, Path]]:
    pat   = re.compile(r'^emb_(.+)_quantized\.tif$')
    tiles = [(m.group(1), f) for f in sorted(ae_dir.iterdir()) if (m := pat.match(f.name))]
    print(f'Test tiles found: {len(tiles)}')
    return tiles


@torch.no_grad()
def run_inference(model, test_tiles, out_dir: Path) -> int:
    model.eval()
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.glob('*.npy'): f.unlink()

    buf_names, buf_ae, buf_prior = [], [], []

    def flush():
        if not buf_names: return
        ae_t  = torch.from_numpy(np.stack(buf_ae)).to(DEVICE)
        pr_t  = torch.from_numpy(np.stack(buf_prior)).to(DEVICE)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            co, ho = model(ae_t, pr_t)
        co = co.float().clamp(0, 1).cpu().numpy()
        ho = ho[:, 0].float().clamp(0).cpu().numpy()
        for i, name in enumerate(buf_names):
            pred = np.concatenate([co[i], ho[i][None]], axis=0).astype(np.float32)
            np.save(out_dir / f'{name}.npy', pred)
        buf_names.clear(); buf_ae.clear(); buf_prior.clear()

    t0 = time.time()
    for i, (tile_name, ae_path) in enumerate(test_tiles):
        ae    = stage1.normalise_ae(ae_path)
        prior = stage1(ae)
        buf_names.append(tile_name)
        buf_ae.append(ae)
        buf_prior.append(np.concatenate([np.clip(prior[:3], 0, 1), np.clip(prior[3:], 0, 60)], 0))
        if len(buf_names) == BATCH_SIZE: flush()
        if (i + 1) % 100 == 0:
            print(f'  {i+1}/{len(test_tiles)}  ({time.time()-t0:.0f}s)', flush=True)
    flush()

    n = len(list(out_dir.glob('*.npy')))
    print(f'Inference done: {n} tiles in {time.time()-t0:.0f}s')
    return n


test_tiles = discover_test_tiles(AE_TEST_DIR)
n = run_inference(model, test_tiles, PRED_DIR)
assert n == 946, f'Expected 946 predictions, got {n}'

## 8 · Package Submission

In [ ]:
def package_submission(pred_dir: Path, zip_path: Path) -> Path:
    files = sorted(pred_dir.glob('*.npy'))
    assert len(files) == 946, f'Expected 946 tiles, got {len(files)}'
    sample = np.load(files[0])
    assert sample.shape == (4, 256, 256) and sample.dtype == np.float32, \
        f'Unexpected shape/dtype: {sample.shape} {sample.dtype}'

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            zf.write(f, arcname=f'predictions/{f.name}')

    size_mb = zip_path.stat().st_size / 1e6
    print(f'Submission: {len(files)} tiles → {zip_path}  ({size_mb:.1f} MB)')
    print('Ready to upload to the competition portal.')
    return zip_path


package_submission(PRED_DIR, SUBMISSION_ZIP)